# Task 3: Window Sum With RNN Encoder And Attention

This task asks you to predict centered window sums. The template uses an RNN encoder and an attention-based decoder.

## TODO

Use PyTorch `torch.nn` modules for the recurrent part.

- Replace the hand-written recurrent cell with `nn.RNN` inside the encoder.
- Keep the attention layers as `nn.Linear` modules.
- For each output position `t`, use the encoder state at position `t` as the attention query.
- Compute attention over all encoder states, form a context vector, and project it to output logits.
- Return logits shaped `(B, T, output_vocab)`.
- If `return_attention=True`, also return attention weights shaped `(B, T, T)`.
- Do not apply `softmax` to the final logits.

In [11]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(217)
K = 2
device

'cpu'

In [12]:
class CustomDataset(Dataset):
    def __init__(self, vocab_size=10, seq_len=25, size=10000, K=2):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.size = size
        self.K = K
        self.X = torch.randint(0, vocab_size, (size, seq_len))
        x_floated = self.X.unsqueeze(1).float()
        kernel = torch.ones((1, 1, 2 * K + 1))
        self.Y = F.conv1d(x_floated, kernel, padding='same').squeeze(1).long()

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

## Model TODO

Complete the classes below. The encoder should use `nn.RNN`; attention can be implemented with batched tensor operations.

In [13]:
class Encoder(nn.Module):
    def __init__(self, input_vocab, d_model, num_layers=1):
        super().__init__()
        # TODO: create nn.Embedding(input_vocab, d_model)
        self.embedding = nn.Embedding(input_vocab, d_model)
        # TODO: create nn.RNN(d_model, d_model, num_layers=num_layers, batch_first=True)
        self.rnn = nn.RNN(d_model, d_model, num_layers=num_layers, batch_first=True)

    def forward(self, x):
        # x: (B, T)
        # TODO: return encoder states H with shape (B, T, d_model)
        embedded = self.embedding(x)
        H, _ = self.rnn(embedded)
        return H

class Attention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.va = nn.Linear(d_model, 1, bias=False)

    def forward(self, query_ht, H):
        # query_ht: (B, d_model), H: (B, T, d_model)
        # TODO: transform query and keys with Wq/Wk
        # TODO: compute Bahdanau scores with tanh and va -> (B, T)
        # TODO: apply softmax over time to get alpha -> (B, T)
        # TODO: compute context vector ct with batch matrix multiply -> (B, d_model)
        # TODO: return ct, alpha
        q = self.Wq(query_ht).unsqueeze(1)
        k = self.Wk(H)
        scores = self.va(torch.tanh(q + k)).squeeze(-1)
        alpha = torch.softmax(scores, dim=-1)
        ct = torch.bmm(alpha.unsqueeze(1), H).squeeze(1)
        return ct, alpha

class Decoder(nn.Module):
    def __init__(self, output_vocab, d_model):
        super().__init__()
        self.attention = Attention(d_model)
        self.Ws = nn.Linear(d_model, d_model, bias=False)
        self.Wc_b = nn.Linear(d_model, d_model, bias=True)
        self.V = nn.Linear(d_model, output_vocab, bias=False)
        self.M_c = nn.Linear(d_model, output_vocab, bias=True)

    def forward(self, H, return_attention=False):
        # H: (B, T, d_model)
        # TODO: loop over t from 0 to T-1
        # TODO: use H[:, t, :] as the query for attention
        # TODO: combine query/context, then project to logits
        # TODO: stack logits into (B, T, output_vocab)
        # TODO: optionally stack attention into (B, T, T)
        B, T, _ = H.shape
        all_logits = []
        all_alphas = []
        for t in range(T):
            query_ht = H[:, t, :]
            ct, alpha = self.attention(query_ht, H)
            combined = torch.tanh(self.Wc_b(ct) + self.Ws(query_ht))
            logits = self.V(combined) + self.M_c(ct)
            all_logits.append(logits)
            all_alphas.append(alpha)
        logits = torch.stack(all_logits, dim=1)
        if return_attention:
            return logits, torch.stack(all_alphas, dim=1)
        return logits

class RNNAttentionModel(nn.Module):
    def __init__(self, K, d_model=128):
        super().__init__()
        self.output_vocab = 9 * (2 * K + 1) + 1
        self.encoder = Encoder(10, d_model)
        self.decoder = Decoder(self.output_vocab, d_model)

    def forward(self, x, return_attention=False):
        # TODO: encode x, then decode H
        H = self.encoder(x)
        return self.decoder(H, return_attention=return_attention)

In [14]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

def train(K=2, epochs=10, batch_size=256):
    model = RNNAttentionModel(K=K).to(device)
    dataset = CustomDataset(K=K)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_set, test_set = random_split(dataset, [train_size, test_size])
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch + 1}: loss={total_loss / len(train_loader):.4f}, acc={evaluate(model, test_loader, device) * 100:.2f}%')
    return model

# Run after completing the model:
model = train(K=K)

Epoch 1: loss=3.0589, acc=8.75%
Epoch 2: loss=2.7992, acc=11.26%
Epoch 3: loss=2.5053, acc=20.11%
Epoch 4: loss=1.9860, acc=38.53%
Epoch 5: loss=1.4818, acc=57.51%
Epoch 6: loss=1.0654, acc=57.61%
Epoch 7: loss=0.8882, acc=75.58%
Epoch 8: loss=0.7242, acc=82.28%
Epoch 9: loss=0.6324, acc=82.80%
Epoch 10: loss=0.5371, acc=84.83%
